<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px 30px; border-radius: 12px; margin-bottom: 10px;">
    <h1 style="color:#e2b96f; font-family:'Segoe UI', sans-serif; font-size:2.2em; margin:0 0 8px 0;">
        🎓 Introdução ao Aprendizado de Máquina
    </h1>
    <h2 style="color:#a8d8ea; font-family:'Segoe UI', sans-serif; font-size:1.3em; margin:0 0 6px 0; font-weight:400;">
        Aula 10 — K-Means e Aprendizado Não Supervisionado
    </h2>
    <h3 style="color:#e2b96f; font-family:'Segoe UI', sans-serif; font-size:1.05em; margin:0 0 12px 0; font-weight:500;">
        🔬 Prática — Descobrindo Grupos Ocultos nos Passageiros do Titanic
    </h3>
    <p style="color:#ccc; font-family:'Segoe UI', sans-serif; font-size:0.9em; margin:0;">
        Prof. Felipe Amaral
    </p>
</div>
<div style="display:flex; gap:10px; margin-top:10px; flex-wrap:wrap;">
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">📚 FIAP</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🐍 Python 3</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🚢 Dataset Titanic</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🔍 Não Supervisionado</span>
</div>


## Uma mudança de paradigma

Ao longo de todo o curso usamos dados **rotulados**: sabíamos que cada passageiro
sobreviveu (1) ou não (0). Nossos modelos aprenderam a **prever esse rótulo**.

Isso é o **aprendizado supervisionado** — temos um professor que nos diz a resposta certa.

Hoje mudamos de paradigma:

> *"E se fingíssemos que não sabemos quem sobreviveu?*
> *Será que o algoritmo consegue descobrir grupos naturais nos dados — sozinho?"*

Esse é o **aprendizado não supervisionado**: sem rótulos, sem professor,
sem resposta certa. O algoritmo descobre a estrutura por conta própria.

---

### O experimento de hoje

1. **Escondemos a coluna `survived`** — o algoritmo não pode vê-la
2. Aplicamos o **K-Means** para encontrar grupos de passageiros similares
3. Depois de formados os grupos, **revelamos os rótulos reais**
4. Analisamos: *os grupos criados pelo algoritmo correspondem a padrões reais?*

---

## Roteiro de hoje

| Parte | Tema |
|-------|------|
| **Config** | Preparando o Titanic para clustering | 
| **1** | Supervisionado vs Não Supervisionado — a diferença fundamental | 
| **2** | Intuição do K-Means — o algoritmo passo a passo | 
| **3** | WCSS e o Método do Cotovelo — escolhendo o K | 
| **4** | Silhouette Score — validando a qualidade dos clusters | 
| **5** | Aplicando K-Means no Titanic — descobrindo os grupos | 
| **6** | Revelando os rótulos — o que o algoritmo descobriu? | 

<div style="background:#d1ecf1; border-left:5px solid #0c5460; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#0c5460;">ℹ️ </strong><span style="color:#0c5460;">O K-Means é a base de aplicações como: segmentação de clientes em e-commerce, agrupamento de documentos, compressão de imagens, detecção de anomalias e recomendação de produtos. É um dos algoritmos mais usados na indústria.</span></div>

---

## Configuração — Execute antes de começar


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.figsize":  (10, 5),
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.titlesize":     13,
    "axes.labelsize":     11,
})
sns.set_theme(style="whitegrid", palette="muted")

# ── Carregando e preparando o Titanic ─────────────────────────────────────────
from sklearn.preprocessing import StandardScaler

df_raw = sns.load_dataset("titanic")
df = df_raw.copy()

# Limpeza
df["age"]      = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df = df.drop(columns=["deck"]).drop_duplicates().reset_index(drop=True)

# Features auxiliares
df["tamanho_familia"]  = df["sibsp"] + df["parch"] + 1
df["sozinho"]          = (df["tamanho_familia"] == 1).astype(int)
df["sex_enc"]          = (df["sex"] == "female").astype(int)
df["pclass_enc"]       = df["pclass"].map({1:2, 2:1, 3:0})
df["faixa_etaria_enc"] = pd.cut(df["age"], bins=[0,12,18,60,100],
                                 labels=[0,1,2,3]).astype(int)
df["titulo"]           = df["name"].str.extract(r",\s([A-Za-z]+)\.")                            .iloc[:,0].map(lambda t: t if t in
                           ["Mr","Miss","Mrs","Master"] else "Raro")
df["tarifa_por_pessoa"]= (df["fare"] / df["tamanho_familia"].clip(lower=1)).round(2)

embarked_ohe = pd.get_dummies(df["embarked"], prefix="embarked", drop_first=True)
titulo_ohe   = pd.get_dummies(df["titulo"],   prefix="titulo",   drop_first=True)
df = pd.concat([df, embarked_ohe, titulo_ohe], axis=1)

# Features para clustering (SEM survived — o algoritmo não pode ver!)
FEATURES_CLUSTER = ["pclass_enc","sex_enc","age","tamanho_familia",
                     "sozinho","faixa_etaria_enc","tarifa_por_pessoa",
                     "embarked_q","embarked_s",
                     "titulo_Master","titulo_Miss","titulo_Mr",
                     "titulo_Mrs","titulo_Raro"]
FEATURES_CLUSTER = [f for f in FEATURES_CLUSTER if f in df.columns]

X_cluster = df[FEATURES_CLUSTER].fillna(0)

# Normalização (K-Means é baseado em distância — obrigatória!)
scaler = StandardScaler()
X_sc   = scaler.fit_transform(X_cluster)

# Guardando os rótulos reais (serão revelados apenas na Parte 6)
y_real = df["survived"].values

print("✅ Dataset pronto para clustering!")
print(f"   {len(df)} passageiros | {len(FEATURES_CLUSTER)} features")
print()
print("IMPORTANTE: a coluna 'survived' foi OCULTADA do algoritmo.")
print("O K-Means não sabe quem sobreviveu — vai descobrir sozinho os grupos.")
print()
print(f"Features usadas para clustering:")
for i, f in enumerate(FEATURES_CLUSTER, 1):
    print(f"  {i:>2}. {f}")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;"><div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;"><div><span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 1</span><h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Supervisionado vs Não Supervisionado</h2><p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"A diferença está em quem conhece a resposta antes de começar."</p></div></div></div>


### Os dois paradigmas

```
SUPERVISIONADO                       NÃO SUPERVISIONADO
─────────────────────────────────    ─────────────────────────────────
Dados:  (X, Y) — com rótulos         Dados: (X) — sem rótulos
Tarefa: aprender X → Y               Tarefa: descobrir estrutura em X
Saída:  previsão para novos X        Saída: grupos, padrões, compressão
Avalia: comparando ŷ com Y real      Avalia: coesão e separação dos grupos

Exemplos:                            Exemplos:
  ✓ Prever sobrevivência (Titanic)     ✓ Segmentar clientes por comportamento
  ✓ Detectar spam                      ✓ Agrupar notícias por tema
  ✓ Reconhecer faces                   ✓ Compressão de imagens
  ✓ Prever preço de imóvel             ✓ Descobrir subtipos de câncer
```

### Tipos de aprendizado não supervisionado

| Técnica | O que faz | Exemplo |
|---------|-----------|---------|
| **Clustering** | Agrupa exemplos similares | K-Means, DBSCAN, Hierárquico |
| **Redução de dimensionalidade** | Simplifica mantendo informação | PCA, t-SNE, UMAP |
| **Detecção de anomalias** | Encontra pontos incomuns | Isolation Forest |
| **Geração** | Cria novos exemplos similares | Autoencoders, GANs |

### Onde o K-Means se encaixa?

O **K-Means** é o algoritmo de **clustering** mais utilizado.
Ele divide os dados em **K grupos** de forma que:
- Pontos do **mesmo grupo** sejam o mais similares possível entre si
- Pontos de **grupos diferentes** sejam o mais diferentes possível


In [ ]:
# Visualizando a diferença: com e sem rótulos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Supervisionado vs Não Supervisionado — Titanic", fontweight="bold")

# Usamos PCA para reduzir para 2D (apenas para visualização)
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_sc)

var_exp = pca.explained_variance_ratio_
print(f"PCA: PC1 explica {var_exp[0]:.1%} | PC2 explica {var_exp[1]:.1%} "
      f"| Total: {sum(var_exp):.1%}")

# Esquerda: sem rótulos (situação do K-Means)
axes[0].scatter(X_2d[:,0], X_2d[:,1],
                color="#0f3460", s=15, alpha=0.4, edgecolors="none")
axes[0].set_title("Situação do K-Means (sem rótulos — só vemos os pontos)",
                  fontweight="bold")
axes[0].set_xlabel(f"PC1 ({var_exp[0]:.1%} da variância)")
axes[0].set_ylabel(f"PC2 ({var_exp[1]:.1%} da variância)")

# Direita: com rótulos reais (informação que escondemos do algoritmo)
cores_surv = ["#e94560" if s == 0 else "#0f3460" for s in y_real]
axes[1].scatter(X_2d[:,0], X_2d[:,1],
                c=cores_surv, s=15, alpha=0.5, edgecolors="none")
axes[1].set_title("Com rótulos reais revelados (informação que o K-Means não tem acesso)",
                  fontweight="bold")
axes[1].set_xlabel(f"PC1 ({var_exp[0]:.1%} da variância)")
axes[1].set_ylabel(f"PC2 ({var_exp[1]:.1%} da variância)")

patch_surv    = mpatches.Patch(color="#0f3460", label="Sobreviveu")
patch_n_surv  = mpatches.Patch(color="#e94560", label="Não Sobreviveu")
axes[1].legend(handles=[patch_surv, patch_n_surv])

plt.tight_layout()
plt.savefig("aula10_sup_vs_naosup.png", dpi=110, bbox_inches="tight")
plt.show()

print("\nDesafio do algoritmo: olhando apenas o gráfico da esquerda,")
print("conseguir descobrir os grupos naturais que vemos na direita?")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 1 — Olhando para o gráfico da esquerda (sem rótulos): (a) você consegue identificar visualmente grupos naturais? Quantos? (b) os dados parecem bem separados ou muito misturados? (c) essa separação sugere que o K-Means terá fácil ou difícil tarefa aqui?</span></div>

*✏️ (a) Grupos visíveis: `???`*

*✏️ (b) Dados estão `???` separados*

*✏️ (c) K-Means terá tarefa `???` porque: `???`*


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;"><div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;"><div><span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 2</span><h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Intuição do K-Means — O Algoritmo Passo a Passo</h2><p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Quatro passos simples que se repetem até convergir."</p></div></div></div>


### O algoritmo

```
1. INICIALIZAÇÃO
   Escolher K centróides aleatoriamente (ou K-Means++)

2. ATRIBUIÇÃO
   Para cada ponto: calcular distância a todos os centróides
   Atribuir o ponto ao centróide mais próximo

3. ATUALIZAÇÃO
   Para cada cluster: recalcular o centróide
   como a MÉDIA de todos os pontos do cluster

4. REPETIR os passos 2 e 3
   até os centróides não se moverem mais (convergência)
```

### O que o algoritmo otimiza?

O K-Means minimiza a **WCSS (Within-Cluster Sum of Squares)** —
a soma das distâncias quadráticas de cada ponto ao centróide do seu cluster:

```
WCSS = Σᵢ Σ_{x ∈ Cᵢ} ‖x − μᵢ‖²

  Cᵢ = cluster i
  μᵢ = centróide do cluster i (média dos pontos de Cᵢ)
```

Quanto menor o WCSS, mais compactos (homogêneos) são os clusters.

<div style="background:#fff3cd; border-left:5px solid #856404; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#856404;">⚠️ </strong><span style="color:#856404;">O K-Means pode convergir para mínimos locais diferentes dependendo da inicialização aleatória dos centróides. O scikit-learn usa <strong>K-Means++</strong> por padrão — uma inicialização inteligente que distribui os centróides iniciais de forma mais espaçada, reduzindo o risco de soluções ruins.</span></div>


In [ ]:
# Implementando K-Means do zero — para entender cada passo
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Dataset 2D simples para visualizar (3 grupos claros)
from sklearn.datasets import make_blobs
X_demo, y_demo = make_blobs(n_samples=120, centers=3,
                              cluster_std=0.8, random_state=42)

def kmeans_manual(X, K, n_iter=6, seed=10):
    "K-Means implementado do zero — retorna histórico de centróides e labels."
    np.random.seed(seed)
    # Inicialização aleatória
    idx = np.random.choice(len(X), K, replace=False)
    centroides = X[idx].copy()

    historico = [centroides.copy()]
    labels_hist = []

    for _ in range(n_iter):
        # Atribuição
        dists  = np.array([[np.linalg.norm(x - c) for c in centroides] for x in X])
        labels = np.argmin(dists, axis=1)
        labels_hist.append(labels.copy())

        # Atualização
        novos = np.array([X[labels == k].mean(axis=0) for k in range(K)])
        centroides = novos
        historico.append(centroides.copy())

    return historico, labels_hist

K = 3
historico, labels_hist = kmeans_manual(X_demo, K)

# Visualizando as primeiras 4 iterações
CORES = ["#0f3460", "#e94560", "#f0a500"]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle("K-Means — Convergência Iteração por Iteração (K=3)",
             fontsize=13, fontweight="bold")

iteracoes = [0, 1, 2, 5]   # quais iterações mostrar
for ax, it in zip(axes, iteracoes):
    labels_it = labels_hist[min(it, len(labels_hist)-1)]
    cent_it   = historico[it]

    # Pontos coloridos por cluster
    for k in range(K):
        mask = labels_it == k
        ax.scatter(X_demo[mask,0], X_demo[mask,1],
                   color=CORES[k], s=30, alpha=0.6, edgecolors="none")

    # Centróides com estrela
    for k in range(K):
        ax.scatter(*cent_it[k], color=CORES[k], s=300,
                   marker="*", edgecolors="white", linewidth=1.5, zorder=10)

    # Centróides anteriores (para mostrar o movimento)
    if it > 0:
        cent_ant = historico[it-1]
        for k in range(K):
            ax.annotate("", xy=cent_it[k], xytext=cent_ant[k],
                        arrowprops=dict(arrowstyle="->", color="#333",
                                        lw=1.5, alpha=0.7))

    titulo = "Inicialização" if it == 0 else f"Iteração {it}"
    ax.set_title(titulo, fontweight="bold")
    ax.set_xlabel("Feature 1"); ax.set_ylabel("Feature 2")
    ax.tick_params(labelsize=8)

plt.tight_layout()
plt.savefig("aula10_kmeans_iteracoes.png", dpi=110, bbox_inches="tight")
plt.show()

print("Observe: os centróides (estrelas) se movem a cada iteração")
print("e param quando chegam ao centro de cada grupo.")


In [ ]:
# Calculando WCSS manualmente para a solução final
labels_final  = labels_hist[-1]
centroides_final = historico[-1]

print("CÁLCULO MANUAL DO WCSS — K-Means do Zero")
print("=" * 50)

wcss_total = 0
for k in range(K):
    pontos_k = X_demo[labels_final == k]
    centroide_k = centroides_final[k]

    # Distâncias quadráticas ao centróide
    dists_k = np.sum((pontos_k - centroide_k)**2, axis=1)
    wcss_k  = dists_k.sum()
    wcss_total += wcss_k

    print(f"Cluster {k}: {len(pontos_k):>3} pontos | "
          f"Centróide: ({centroide_k[0]:+.2f}, {centroide_k[1]:+.2f}) | "
          f"WCSS parcial: {wcss_k:.2f}")

print(f"{'─'*55}")
print(f"WCSS Total: {wcss_total:.2f}")
print()

# Verificando com sklearn
from sklearn.cluster import KMeans
km_check = KMeans(n_clusters=3, random_state=42, n_init=10)
km_check.fit(X_demo)
print(f"Verificação sklearn (inertia_): {km_check.inertia_:.2f}")
print(f"(sklearn usa K-Means++ — resultado pode diferir ligeiramente)")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 2 — Analise as 4 iterações visualizadas e responda: (a) na inicialização, os centróides já estão nos centros dos grupos? (b) após quantas iterações os centróides parecem ter convergido? (c) o que aconteceria se inicializássemos os 3 centróides no mesmo ponto?</span></div>

*✏️ (a) Na inicialização: `???`*

*✏️ (b) Convergência após ~`???` iterações*

*✏️ (c) Com centróides iguais: `???` porque: `???`*


In [ ]:
# ── GABARITO DA MISSÃO 2 (descomente para ver) ───────────────────────────────
# print("Gabarito:")
# print()
# print("(a) Na inicializacao aleatoria, os centróides raramente estao nos centros reais.")
# print("    O K-Means++ (padrao sklearn) distribui melhor os centróides iniciais.")
# print("    A convergencia parte de longe do otimo — mas chega la em poucas iteracoes.")
# print()
# print("(b) Para 3 grupos bem separados, K-Means converge em 3-6 iteracoes.")
# print("    Para grupos sobrepostos, pode levar mais iteracoes ou nao convergir bem.")
# print()
# print("(c) Se os 3 centróides comecar no mesmo ponto:")
# print("    As distancias de todos os pontos aos 3 centróides serao iguais.")
# print("    A atribuicao sera aleatoria/deterministica mas identica para todos.")
# print("    Os centróides atualizados serao identicos entre si.")
# print("    O algoritmo nao converge — ficara com 3 clusters identicos.")
# print("    Por isso o scikit-learn inicializa com K-Means++ e tenta n_init=10 vezes.")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;"><div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;"><div><span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 3</span><h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">WCSS e o Método do Cotovelo — Escolhendo o K</h2><p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"O maior desafio do K-Means: quantos grupos existem nos dados?"</p></div></div></div>


### O problema: K não é conhecido

O K-Means exige que você diga quantos grupos existem — mas em dados reais,
esse número geralmente não é conhecido. Como escolher?

### Método do Cotovelo (Elbow Method)

Treinamos o K-Means para K = 1, 2, 3, ..., n e plotamos o WCSS.

```
WCSS
│
│ ●
│   ●
│      ●  ← cotovelo (ponto de inflexão)
│         ● ● ● ● ●
└──────────────────── K
   1  2  3  4  5  6
```

**O "cotovelo"** é o ponto onde a redução do WCSS começa a diminuir
significativamente — adicionar mais clusters não traz ganho proporcional.

### Interpretação

- K muito pequeno → WCSS alto (clusters muito grandes e heterogêneos)
- K muito grande → WCSS baixo mas clusters sem sentido (cada ponto é um cluster)
- K ideal → equilíbrio entre compactação e interpretabilidade


In [ ]:
from sklearn.cluster import KMeans

# Calculando WCSS para K de 1 a 10 — no Titanic
Ks    = range(1, 11)
wcss  = []

for k in Ks:
    km = KMeans(n_clusters=k, init="k-means++",
                n_init=10, random_state=42, max_iter=300)
    km.fit(X_sc)
    wcss.append(km.inertia_)
    print(f"  K={k:>2} | WCSS = {km.inertia_:>10.2f}", end="")
    if k > 1:
        reducao = (wcss[-2] - wcss[-1]) / wcss[-2] * 100
        print(f" | Redução: {reducao:.1f}%", end="")
    print()

print()
print("Observe: a redução de WCSS diminui com K crescente.")
print("O 'cotovelo' sugere o K ideal.")


In [ ]:
# Visualizando o método do cotovelo
import numpy as np

# Encontrando o cotovelo automaticamente (método da segunda derivada)
wcss_arr = np.array(wcss)
d1 = np.diff(wcss_arr)          # primeira derivada (quedas)
d2 = np.diff(d1)                 # segunda derivada (aceleração da queda)
cotovelo_idx = np.argmax(np.abs(d2)) + 2   # +2 por causa de dois diffs
K_cotovelo   = list(Ks)[cotovelo_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Método do Cotovelo — Escolhendo K para o Titanic",
             fontweight="bold")

# Gráfico 1: WCSS por K
axes[0].plot(Ks, wcss, "o-", color="#0f3460", linewidth=2.5, markersize=8)
axes[0].axvline(K_cotovelo, color="#e94560", linestyle="--",
                linewidth=2, label=f"Cotovelo K={K_cotovelo}")
axes[0].fill_betweenx([min(wcss)*0.9, max(wcss)*1.05],
                       K_cotovelo-0.3, K_cotovelo+0.3,
                       alpha=0.12, color="#e94560")

# Anotando a redução percentual
for i in range(1, len(Ks)):
    reducao = (wcss[i-1] - wcss[i]) / wcss[i-1] * 100
    cor = "#e94560" if reducao < 10 else "#0f3460"
    axes[0].annotate(f"-{reducao:.0f}%",
                     xy=(list(Ks)[i], wcss[i]),
                     xytext=(list(Ks)[i]+0.15, wcss[i] + max(wcss)*0.02),
                     fontsize=8, color=cor)

axes[0].set_xlabel("Número de Clusters (K)")
axes[0].set_ylabel("WCSS (Inércia)")
axes[0].set_title("WCSS por K (cotovelo = ponto de inflexão)")
axes[0].legend()

# Gráfico 2: variação percentual do WCSS
reducoes = [(wcss[i-1]-wcss[i])/wcss[i-1]*100 for i in range(1, len(Ks))]
axes[1].bar(list(Ks)[1:], reducoes, color=["#e94560" if r < 10 else "#0f3460"
                                             for r in reducoes],
            edgecolor="white", width=0.6)
axes[1].axhline(10, color="#f0a500", linestyle="--",
                linewidth=1.5, label="Limiar 10%")
axes[1].set_xlabel("K")
axes[1].set_ylabel("Redução do WCSS (%)")
axes[1].set_title("Redução Percentual do WCSS (abaixo de 10% → ganho marginal)",
                  fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.savefig("aula10_cotovelo.png", dpi=110, bbox_inches="tight")
plt.show()

print(f"Cotovelo detectado automaticamente: K = {K_cotovelo}")
print(f"Isso sugere que os dados do Titanic têm ~{K_cotovelo} grupos naturais.")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 3 — Analise o gráfico do cotovelo e responda: (a) qual K você escolheria olhando apenas para o gráfico? Justifique. (b) o cotovelo detectado automaticamente (K=2) faz sentido para o contexto do Titanic? O que você esperaria encontrar nesses grupos? (c) por que não escolher K=10 se o WCSS é menor?</span></div>

*✏️ (a) Escolheria K = `???` porque: `???`*

*✏️ (b) K = `???` faz sentido para o Titanic? `???`*

*✏️ (c) Não escolhemos K=10 porque: `???`*


In [ ]:
# ── GABARITO DA MISSÃO 3 (descomente para ver) ───────────────────────────────
# print("Gabarito:")
# print()
# print("(a) O cotovelo geralmente cai entre K=2 e K=4 no Titanic.")
# print("    K=2 é o mais óbvio: sobreviveu vs nao sobreviveu.")
# print("    K=3 ou K=4 podem capturar subdivisoes: classe social, genero, etc.")
# print()
# print("(b) K=2 ou K=3 faz muito sentido para o Titanic:")
# print("    K=2: grupo de alta sobrevivencia (mulheres, 1a classe)")
# print("         vs grupo de baixa sobrevivencia (homens, 3a classe)")
# print("    K=3: adiciona um grupo intermediario (2a classe, homens de 1a classe)")
# print()
# print("(c) Nao escolhemos K=10 porque:")
# print("    WCSS sempre cai com mais clusters (ate K=n cada ponto e um cluster).")
# print("    Clusters demais sao dificeis de interpretar e perdem significado.")
# print("    O objetivo e encontrar grupos UTEIS e INTERPRETAVEIS, nao minimizar WCSS.")
# print("    Com K=10 teriamos 10 perfis de passageiro — impossivel de usar na pratica.")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;"><div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;"><div><span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 4</span><h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Silhouette Score — Validando a Qualidade dos Clusters</h2><p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"O cotovelo sugere K. O Silhouette confirma se os grupos são realmente bons."</p></div></div></div>


### Além do WCSS: qualidade interna dos clusters

O WCSS diz o quanto os pontos são próximos do **centróide do seu próprio cluster**,
mas não mede se os clusters estão bem **separados entre si**.

O **Silhouette Score** mede as duas coisas ao mesmo tempo, para cada ponto:

```
         b(i) − a(i)
s(i) = ────────────────
          max(a(i), b(i))

onde:
  a(i) = distância média de i a todos os outros pontos do SEU cluster
          (coesão — quanto menor, melhor)

  b(i) = distância média de i aos pontos do cluster VIZINHO mais próximo
          (separação — quanto maior, melhor)
```

### Interpretando o Silhouette Score

| Valor | Significado |
|-------|-------------|
| **≈ +1** | Ponto bem dentro do cluster, longe dos outros |
| **≈ 0** | Ponto na fronteira entre dois clusters |
| **≈ −1** | Ponto provavelmente no cluster errado |

O **Silhouette Score médio** do dataset deve ser o mais próximo de +1 possível.


In [ ]:
from sklearn.metrics import silhouette_score, silhouette_samples

# Calculando Silhouette Score para diferentes valores de K
print("SILHOUETTE SCORE por K — Titanic")
print("=" * 45)
print(f"  {'K':>4}  {'Silhouette':>12}  {'WCSS':>12}  {'Avaliação'}")
print("  " + "-"*44)

sil_scores = []
wcss_scores = []

for k in range(2, 9):
    km = KMeans(n_clusters=k, init="k-means++",
                n_init=10, random_state=42)
    labels_k = km.fit_predict(X_sc)
    sil  = silhouette_score(X_sc, labels_k)
    inertia = km.inertia_
    sil_scores.append(sil)
    wcss_scores.append(inertia)

    if sil > 0.35:    avaliacao = "✅ Bom"
    elif sil > 0.20:  avaliacao = "⚠️ Aceitável"
    else:             avaliacao = "❌ Fraco"

    print(f"  {k:>4}  {sil:>12.4f}  {inertia:>12.1f}  {avaliacao}")

melhor_K_sil = list(range(2, 9))[np.argmax(sil_scores)]
print()
print(f"K com maior Silhouette Score: {melhor_K_sil} ({max(sil_scores):.4f})")


In [ ]:
# Diagrama de Silhouette para o K ideal
import matplotlib.cm as cm

K_viz = melhor_K_sil
km_viz = KMeans(n_clusters=K_viz, init="k-means++",
                n_init=10, random_state=42)
labels_viz = km_viz.fit_predict(X_sc)
sil_vals   = silhouette_samples(X_sc, labels_viz)
sil_medio  = sil_vals.mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f"Silhouette Analysis — K={K_viz}  |  Score médio: {sil_medio:.4f}",
             fontsize=13, fontweight="bold")

# Diagrama de silhouette
y_lower = 10
cmap    = cm.get_cmap("tab10")

for k in range(K_viz):
    vals_k   = np.sort(sil_vals[labels_viz == k])
    size_k   = len(vals_k)
    y_upper  = y_lower + size_k
    cor      = cmap(k / K_viz)
    ax1.fill_betweenx(np.arange(y_lower, y_upper), 0, vals_k,
                       facecolor=cor, alpha=0.75)
    ax1.text(-0.05, y_lower + 0.5*size_k,
             f"C{k} (n={size_k})", fontsize=9)
    y_lower  = y_upper + 10

ax1.axvline(sil_medio, color="#e94560", linestyle="--",
            linewidth=2, label=f"Score médio = {sil_medio:.3f}")
ax1.set_xlabel("Silhouette Coefficient")
ax1.set_ylabel("Cluster")
ax1.set_title(f"Diagrama de Silhouette (K={K_viz}) "
              "Barras largas e à direita = clusters coesos e separados")
ax1.legend()

# Comparação visual dos clusters no espaço PCA
for k in range(K_viz):
    mask = labels_viz == k
    ax2.scatter(X_2d[mask, 0], X_2d[mask, 1],
                color=cmap(k / K_viz), s=20, alpha=0.6,
                label=f"Cluster {k} (n={mask.sum()})", edgecolors="none")

# Centróides projetados
centroides_2d = pca.transform(km_viz.cluster_centers_)
ax2.scatter(centroides_2d[:,0], centroides_2d[:,1],
            color="white", s=250, marker="*",
            edgecolors="black", linewidth=1.5, zorder=10, label="Centróides")

ax2.set_xlabel(f"PC1 ({var_exp[0]:.1%})")
ax2.set_ylabel(f"PC2 ({var_exp[1]:.1%})")
ax2.set_title(f"Clusters no Espaço PCA (K={K_viz})", fontweight="bold")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig("aula10_silhouette.png", dpi=110, bbox_inches="tight")
plt.show()


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 4 — Interprete o diagrama de Silhouette: (a) qual cluster tem a barra mais larga (mais coeso)? O que isso significa? (b) há algum cluster com valores negativos de Silhouette? O que indicaria? (c) o Score médio indica clusters bons, aceitáveis ou fracos?</span></div>

*✏️ (a) Cluster mais coeso: `???` — isso significa: `???`*

*✏️ (b) Valores negativos: `???` — indicaria: `???`*

*✏️ (c) Score médio de `???` indica clusters `???`*


In [ ]:
# ── GABARITO DA MISSÃO 4 (descomente para ver) ───────────────────────────────
# print("Gabarito:")
# print()
# print("(a) O cluster mais coeso tem a barra mais uniforme e extensa para a direita.")
# print("    Isso significa que os passageiros desse grupo sao muito similares entre si.")
# print("    Provavelmente corresponde a um grupo bem definido como 'mulheres de 1a classe'.")
# print()
# print("(b) Valores negativos indicam pontos no cluster ERRADO.")
# print("    Esses passageiros estao mais proximos de outro cluster do que do seu atual.")
# print("    No Titanic, passageiros 'intermediarios' (ex: homem de 2a classe) podem")
# print("    ter silhouette negativo por estarem entre dois grupos.")
# print()
# s_med = sil_vals.mean()
# if s_med > 0.35:
#     avaliacao = "BOM — clusters claramente separados"
# elif s_med > 0.20:
#     avaliacao = "ACEITAVEL — alguma sobreposicao entre clusters"
# else:
#     avaliacao = "FRACO — clusters muito sobrepostos"
# print(f"(c) Score medio = {s_med:.4f} — {avaliacao}")
# print("    Dados humanos raramente passam de 0.5 em Silhouette — a sobreposicao e esperada.")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;"><div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;"><div><span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 5</span><h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Aplicando K-Means no Titanic — Descobrindo os Grupos</h2><p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Com K definido, vamos treinar e analisar o que o algoritmo encontrou."</p></div></div></div>


### Escolhendo o K final

Vamos usar os dois métodos juntos para decidir:

| Método | Sugestão |
|--------|----------|
| Cotovelo | K = indicado pelo gráfico |
| Silhouette máximo | K = indicado pela análise |
| Conhecimento do domínio | Titanic tem grupos naturais? |

Testamos K=2, K=3 e K=4 — os mais indicados pelos métodos acima —
e analisamos qual gera clusters mais interpretáveis.


In [ ]:
# Treinando K-Means para K=2, K=3 e K=4
resultados_K = {}

for k in [2, 3, 4]:
    km = KMeans(n_clusters=k, init="k-means++",
                n_init=10, random_state=42)
    labels = km.fit_predict(X_sc)
    resultados_K[k] = {
        "modelo":  km,
        "labels":  labels,
        "wcss":    km.inertia_,
        "sil":     silhouette_score(X_sc, labels),
    }

print("COMPARAÇÃO DE K=2, K=3 e K=4")
print("=" * 50)
print(f"  {'K':>3}  {'WCSS':>12}  {'Silhouette':>12}  {'Clusters formados'}")
print("  " + "-"*48)
for k, res in resultados_K.items():
    tamanhos = [f"C{i}={np.sum(res['labels']==i)}" for i in range(k)]
    print(f"  {k:>3}  {res['wcss']:>12.1f}  {res['sil']:>12.4f}  "
          f"  {', '.join(tamanhos)}")


In [ ]:
# Visualizando os 3 soluções lado a lado no espaço PCA
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("K-Means no Titanic — K=2, K=3 e K=4 (Espaço PCA 2D)",
             fontsize=13, fontweight="bold")

for ax, (k, res) in zip(axes, resultados_K.items()):
    labels_k  = res["labels"]
    centro_2d = pca.transform(res["modelo"].cluster_centers_)
    cmap_k    = cm.get_cmap("tab10")

    for c in range(k):
        mask = labels_k == c
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                   color=cmap_k(c/k), s=15, alpha=0.55, edgecolors="none",
                   label=f"C{c} (n={mask.sum()})")

    ax.scatter(centro_2d[:,0], centro_2d[:,1],
               color="white", s=300, marker="*",
               edgecolors="black", linewidth=2, zorder=10)

    ax.set_title(f"K={k}  |  WCSS={res['wcss']:.0f}  "
                 f"|  Sil={res['sil']:.3f}", fontweight="bold")
    ax.set_xlabel(f"PC1 ({var_exp[0]:.1%})")
    ax.set_ylabel(f"PC2 ({var_exp[1]:.1%})")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("aula10_kmeans_K_comparacao.png", dpi=110, bbox_inches="tight")
plt.show()


In [ ]:
# Analisando o perfil de cada cluster — K=3 (geralmente mais informativo)
K_escolhido = 3
labels_final = resultados_K[K_escolhido]["labels"]

# Adicionando os clusters ao dataframe para análise
df_analise = df.copy()
df_analise["cluster"] = labels_final

print(f"PERFIL DOS CLUSTERS — K={K_escolhido}")
print("=" * 65)

features_analise = ["pclass","sex","age","tamanho_familia",
                    "fare","sozinho","survived"]

for c in range(K_escolhido):
    sub = df_analise[df_analise["cluster"] == c]
    print(f"\nCLUSTER {c}  (n={len(sub)}, {len(sub)/len(df_analise):.0%} dos passageiros)")
    print("-" * 50)

    # Estatísticas numéricas
    print(f"  Classe (média):      {sub['pclass'].mean():.2f}  "
          f"(1=rica, 3=pobre)")
    print(f"  Feminino:            {sub['sex_enc'].mean():.0%}")
    print(f"  Idade média:         {sub['age'].mean():.1f} anos")
    print(f"  Família (média):     {sub['tamanho_familia'].mean():.1f} pessoas")
    print(f"  Tarifa média:        £{sub['fare'].mean():.2f}")
    print(f"  Viajando sozinho:    {sub['sozinho'].mean():.0%}")
    print()
    # A grande revelação — mas deixamos para a Parte 6!
    print(f"  Taxa de sobreviv.:   *** REVELADA NA PARTE 6 ***")


In [ ]:
# Heatmap visual dos perfis dos clusters
# (sem revelar survived ainda)
features_heatmap = ["pclass_enc","sex_enc","age","tamanho_familia",
                    "sozinho","tarifa_por_pessoa","faixa_etaria_enc"]

# Média normalizada de cada feature por cluster
perfis = []
for c in range(K_escolhido):
    mask  = labels_final == c
    media = X_sc[mask].mean(axis=0)[:len(features_heatmap)]
    perfis.append(media)

df_perfis = pd.DataFrame(perfis,
                          columns=features_heatmap,
                          index=[f"Cluster {c}" for c in range(K_escolhido)])

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(df_perfis,
            annot=True, fmt=".2f",
            cmap="RdYlBu", center=0,
            vmin=-2, vmax=2,
            linewidths=0.5,
            ax=ax,
            annot_kws={"size": 10})
ax.set_title(f"Perfil dos Clusters (K={K_escolhido}) — Médias Normalizadas"
             "Azul = acima da média global | Vermelho = abaixo",
             fontweight="bold")
ax.set_xlabel("Features"); ax.set_ylabel("Cluster")
plt.tight_layout()
plt.savefig("aula10_heatmap_perfis.png", dpi=110, bbox_inches="tight")
plt.show()

print("Interpretação das cores:")
print("  Azul escuro (>0): cluster tem valor ACIMA da média global")
print("  Vermelho escuro (<0): cluster tem valor ABAIXO da média global")
print()
print("Tente nomear os clusters baseado nos perfis — antes de ver survived!")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 5 — Baseado no heatmap e nas estatísticas acima, tente nomear cada cluster em termos de perfil de passageiro. Depois, faça uma previsão: qual cluster você acha que tem maior taxa de sobrevivência? Registre abaixo antes de ir para a Parte 6.</span></div>

*✏️ Meu nome para o Cluster 0: `???` (ex: 'homens jovens de 3ª classe')*

*✏️ Meu nome para o Cluster 1: `???`*

*✏️ Meu nome para o Cluster 2: `???`*

*✏️ Previsão de taxa de sobrevivência (maior para menor): Cluster `???` > `???` > `???`*


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;"><div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;"><div><span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 6</span><h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Revelando os Rótulos — O que o Algoritmo Descobriu?</h2><p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"O momento da verdade: os grupos criados correspondem à sobrevivência?"</p></div></div></div>


### A grande revelação

O K-Means nunca viu a coluna `survived`.
Ele agrupou passageiros apenas pela **similaridade das características**.

Agora vamos descobrir se os grupos que ele criou corresponderam — ou não —
aos padrões de sobrevivência que analisamos ao longo de todo o curso.


In [ ]:
# ── A GRANDE REVELAÇÃO ────────────────────────────────────────────────────────
print("=" * 60)
print("  REVELAÇÃO FINAL — Taxa de Sobrevivência por Cluster")
print("=" * 60)

for c in range(K_escolhido):
    sub = df_analise[df_analise["cluster"] == c]
    taxa = sub["survived"].mean()
    genero_f = sub["sex_enc"].mean()
    classe   = sub["pclass"].mean()

    barra = "█" * int(taxa * 30)
    print(f"\nCluster {c}  (n={len(sub):>3})")
    print(f"  Perfil:            "
          f"{'Mais feminino' if genero_f > 0.5 else 'Mais masculino'}, "
          f"Classe média: {classe:.1f}")
    print(f"  Sobrevivência:     {taxa:.1%}  {barra}")
    print(f"  {'🟢 Alta sobrevivência' if taxa > 0.5 else ('🟡 Média' if taxa > 0.3 else '🔴 Baixa sobrevivência')}")


In [ ]:
# Visualização completa: clusters coloridos com revelação de survived
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("A Revelação — Clusters K-Means vs Sobrevivência Real",
             fontsize=13, fontweight="bold")

cmap_k = cm.get_cmap("tab10")

# Gráfico 1: clusters do K-Means
for c in range(K_escolhido):
    mask = labels_final == c
    taxa = df_analise[df_analise["cluster"] == c]["survived"].mean()
    axes[0].scatter(X_2d[mask, 0], X_2d[mask, 1],
                    color=cmap_k(c/K_escolhido), s=15, alpha=0.6,
                    edgecolors="none",
                    label=f"C{c} — sobrev.: {taxa:.0%}")
centro_2d = pca.transform(resultados_K[K_escolhido]["modelo"].cluster_centers_)
axes[0].scatter(centro_2d[:,0], centro_2d[:,1],
                color="white", s=300, marker="*",
                edgecolors="black", linewidth=2, zorder=10)
axes[0].set_title("Clusters do K-Means (cores = grupos criados pelo algoritmo)",
                  fontweight="bold")
axes[0].set_xlabel(f"PC1 ({var_exp[0]:.1%})")
axes[0].set_ylabel(f"PC2 ({var_exp[1]:.1%})")
axes[0].legend(fontsize=9)

# Gráfico 2: sobrevivência real
for sobrev, cor, lbl in [(0, "#e94560", "Não Sobreviveu"),
                          (1, "#0f3460", "Sobreviveu")]:
    mask = y_real == sobrev
    axes[1].scatter(X_2d[mask, 0], X_2d[mask, 1],
                    color=cor, s=15, alpha=0.5,
                    edgecolors="none", label=lbl)
axes[1].set_title("Rótulos Reais de Sobrevivência "
                  "(azul=sobreviveu, vermelho=não sobreviveu)",
                  fontweight="bold")
axes[1].set_xlabel(f"PC1 ({var_exp[0]:.1%})")
axes[1].set_ylabel(f"PC2 ({var_exp[1]:.1%})")
axes[1].legend()

# Gráfico 3: barras de sobrevivência por cluster
taxas = [df_analise[df_analise["cluster"]==c]["survived"].mean()
         for c in range(K_escolhido)]
ns    = [df_analise[df_analise["cluster"]==c].shape[0]
         for c in range(K_escolhido)]
cores_barra = [cmap_k(c/K_escolhido) for c in range(K_escolhido)]
barras = axes[2].bar([f"Cluster {c} (n={ns[c]})" for c in range(K_escolhido)],
                      taxas, color=cores_barra, edgecolor="white", width=0.5)
axes[2].axhline(df["survived"].mean(), color="gray", linestyle="--",
                linewidth=1.5, label=f"Média global ({df['survived'].mean():.0%})")
for b, v in zip(barras, taxas):
    axes[2].text(b.get_x()+b.get_width()/2, v+0.02,
                 f"{v:.0%}", ha="center", fontsize=12, fontweight="bold")
axes[2].set_ylabel("Taxa de Sobrevivência")
axes[2].set_title("Taxa de Sobrevivência por Cluster"
                  "(K-Means não tinha acesso a essa informação)",
                  fontweight="bold")
axes[2].set_ylim(0, 1.0); axes[2].legend()

plt.tight_layout()
plt.savefig("aula10_revelacao_final.png", dpi=110, bbox_inches="tight")
plt.show()


In [ ]:
# Avaliação quantitativa: quão bem o K-Means recuperou os padrões?
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# Adjusted Rand Index: 0 = aleatório, 1 = perfeito
ari = adjusted_rand_score(y_real, labels_final)

# Normalized Mutual Information: 0 = independente, 1 = perfeito
nmi = normalized_mutual_info_score(y_real, labels_final)

print("AVALIAÇÃO QUANTITATIVA — Clusters vs Sobrevivência Real")
print("=" * 58)
print()
print(f"  Adjusted Rand Index (ARI): {ari:.4f}")
print(f"    → 0 = aleatorio | 1 = recuperacao perfeita dos rotulos")
print()
print(f"  Normalized Mutual Information (NMI): {nmi:.4f}")
print(f"    → 0 = independente | 1 = informacao total sobre os rotulos")
print()

if ari > 0.3:
    conclusao = "O K-Means recuperou bem os padroes de sobrevivencia!"
elif ari > 0.1:
    conclusao = "O K-Means capturou parcialmente os padroes de sobrevivencia."
else:
    conclusao = "Os clusters do K-Means sao fracos preditores de sobrevivencia."

print(f"  Conclusao: {conclusao}")
print()
print("Reflexao importante:")
print("  O K-Means NUNCA viu 'survived' — agrupou apenas por similaridade.")
print("  Se os clusters se alinham com sobrevivencia, e porque os determinantes")
print("  da sobrevivencia (genero, classe, idade) tambem criam grupos naturais!")


In [ ]:
# Tabela cruzada: cluster vs sobrevivência real
print("TABELA CRUZADA — Cluster vs Sobrevivência")
print()
crosstab = pd.crosstab(
    df_analise["cluster"],
    df_analise["survived"],
    rownames=["Cluster"],
    colnames=["Sobreviveu (real)"],
    margins=True,
    margins_name="Total"
)
crosstab.columns = ["Nao Sobreviveu", "Sobreviveu", "Total"]
print(crosstab.to_string())
print()

# Adicionando proporção
crosstab_prop = pd.crosstab(
    df_analise["cluster"],
    df_analise["survived"],
    normalize="index"
).round(3) * 100
crosstab_prop.columns = ["% Nao Sobreviveu", "% Sobreviveu"]
print("Proporção por cluster (%):")
print(crosstab_prop.to_string())


In [ ]:
# ── GABARITO DA MISSÃO 5 (descomente para ver) ───────────────────────────────
# print("Gabarito esperado para K=3 no Titanic:")
# print()
# for c in range(K_escolhido):
#     sub  = df_analise[df_analise["cluster"] == c]
#     taxa = sub["survived"].mean()
#     genero = sub["sex_enc"].mean()
#     classe = sub["pclass"].mean()
#     idade  = sub["age"].mean()
#     print(f"Cluster {c}:")
#     print(f"  Genero: {genero:.0%} feminino | Classe: {classe:.1f} | "
#           f"Idade: {idade:.0f} anos")
#     print(f"  Sobrevivência: {taxa:.0%}")
#
#     if taxa > 0.6:
#         print(f"  Nome: 'Grupo de alta sobrevivencia'")
#         print(f"  → Mulheres e criancas, principalmente 1a e 2a classe")
#     elif taxa > 0.3:
#         print(f"  Nome: 'Grupo intermediario'")
#         print(f"  → Mix de perfis — 1a classe masculina, mulheres 3a classe")
#     else:
#         print(f"  Nome: 'Grupo de baixa sobrevivencia'")
#         print(f"  → Homens adultos, principalmente 3a classe")
#     print()


---

## Resumo — O que o K-Means encontrou no Titanic?

O experimento revelou algo fascinante:
**o K-Means, sem nunca ter visto quem sobreviveu, criou grupos que se alinham
com os padrões de sobrevivência** — porque as mesmas variáveis que determinaram
a sobrevivência (gênero, classe, tarifa) também criam grupos naturais nos dados.

Isso é o poder do aprendizado não supervisionado: **revelar estrutura que existe
nos dados, independentemente de qualquer rótulo**.

---

## Checklist — O que você sabe fazer agora

| Habilidade | Praticada hoje? |
|------------|----------------|
| Distinguir aprendizado supervisionado de não supervisionado | ☐ |
| Implementar o K-Means do zero (inicialização, atribuição, atualização) | ☐ |
| Calcular WCSS manualmente | ☐ |
| Usar o Método do Cotovelo para escolher K | ☐ |
| Calcular e interpretar o Silhouette Score | ☐ |
| Ler o Diagrama de Silhouette por cluster | ☐ |
| Treinar K-Means com scikit-learn e analisar os clusters | ☐ |
| Criar e interpretar o heatmap de perfis de cluster | ☐ |
| Avaliar clusters com ARI e NMI comparando com rótulos externos | ☐ |

---


## Reflexão final

<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">Escreva em suas próprias palavras: (1) qual é a diferença fundamental entre classificar e fazer clustering? (2) por que o K-Means precisa de normalização? (3) em que situação real de negócio você usaria K-Means em vez de um modelo supervisionado?</span></div>


**✏️ Minha reflexão:**

1. Classificar vs Clustering: *...*

2. K-Means precisa de normalização porque: *...*

3. Usaria K-Means (não supervisionado) em vez de modelo supervisionado quando: *...*


---

## O que vem a seguir?

```
Supervisionado:     KNN ✅  Log. Reg. ✅  SVM ✅  Árvore ✅
Regressão:          Regressão Linear ✅
Não Supervisionado: K-Means ✅
Próximos:           Avaliação completa de modelos  →  Deploy e MLOps
```

O K-Means é apenas um dos algoritmos de clustering disponíveis.
Alternativas importantes incluem DBSCAN (que encontra clusters de formas arbitrárias
e detecta outliers automaticamente) e Clustering Hierárquico (que não exige K definido).

---

## Referências

- Scikit-Learn KMeans: https://scikit-learn.org/stable/modules/clustering.html
- MacQueen, J. (1967). *Some methods for classification and analysis of multivariate observations*. Berkeley Symposium.
- Rousseeuw, P. J. (1987). *Silhouettes: a graphical aid to the interpretation and validation of cluster analysis*.
- Géron, A. (2019). *Hands-on Machine Learning*, Cap. 9. O'Reilly.
